In [29]:
import pandas as pd
import numpy as np
import re
import os

In [28]:
EU_COUNTRIES = [
    'BE', 'BG', 'CZ', 'DK', 'DE', 'EE', 'IE', 
    'EL', 'ES', 'FR', 'HR', 'IT', 'CY', 'LV',
    'LT', 'LU', 'HU', 'MT', 'NL', 'AT', 'PL', 
    'PT', 'RO', 'SI', 'SK', 'FI', 'SE'
]

EFTA_COUNTRIES = ['IS', 'LI', 'NO', 'CH']

EU_EFTA = EU_COUNTRIES + EFTA_COUNTRIES

### AI adoption rate - 2023

The dependent variable is the proportion of firms within each country - sector using at least one type of AI in 2023. Unit of measurement - share of the firms. PC_ENT

In [30]:
ai_adopt = pd.read_csv('C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/processed data/ain2.csv', index_col=0)
print(ai_adopt)

       freq size_emp nace_r2           indic_is    unit geo  num_2021  \
0         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT  AT       NaN   
1         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT  BA       NaN   
2         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT  BE       NaN   
3         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT  BG       NaN   
4         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT  CY       NaN   
...     ...      ...     ...                ...     ...  ..       ...   
221987    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT  RO       NaN   
221988    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT  RS       NaN   
221989    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT  SE       NaN   
221990    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT  SI       NaN   
221991    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT  SK       NaN   

       flag_2021  num_2023 flag_2023  num_2024 flag_2024  
0            NaN      8.24       NaN       NaN       NaN  
1    

In [31]:
# Filter the data 
# E_AI_TANY - Enterprises use at least one of the AI technologies

ai_adopt_2023 = ai_adopt.copy()
ai_adopt_2023 = ai_adopt.drop(columns=[ 'freq', 'size_emp', 'num_2021', 'flag_2021', 'flag_2023', 'num_2024', 'flag_2024'])

ai_adopt_2023 = ai_adopt_2023[
    (ai_adopt_2023["indic_is"] == "E_AI_TANY") &
    (ai_adopt_2023["unit"] == "PC_ENT") 
]

ai_adopt_2023 = ai_adopt_2023[ai_adopt_2023['geo'].isin(EU_EFTA)]
print(ai_adopt_2023)

       nace_r2   indic_is    unit geo  num_2023
3809         C  E_AI_TANY  PC_ENT  AT     12.31
3811         C  E_AI_TANY  PC_ENT  BE     15.31
3812         C  E_AI_TANY  PC_ENT  BG      2.55
3813         C  E_AI_TANY  PC_ENT  CY      3.81
3814         C  E_AI_TANY  PC_ENT  CZ      6.01
...        ...        ...     ...  ..       ...
221156    S951  E_AI_TANY  PC_ENT  PT      8.82
221157    S951  E_AI_TANY  PC_ENT  RO      0.00
221159    S951  E_AI_TANY  PC_ENT  SE      6.67
221160    S951  E_AI_TANY  PC_ENT  SI       NaN
221161    S951  E_AI_TANY  PC_ENT  SK      0.00

[1339 rows x 5 columns]


In [32]:
def nace_section_or_nan(s: str) -> str | float:
    s = str(s)
    m = re.match(r'^([A-U])', s)
    if not m:
        return np.nan
    first = m.group(1)

    if re.search(r'-([A-U])', s) and re.search(r'-([A-U])', s).group(1) != first:
        return np.nan

    if re.search(r'_([A-U])', s) and re.search(r'_([A-U])', s).group(1) != first:
        return np.nan

    return first

ai_adopt_2023['nace_r2_1d'] = ai_adopt_2023['nace_r2'].map(nace_section_or_nan)
print(ai_adopt_2023)

       nace_r2   indic_is    unit geo  num_2023 nace_r2_1d
3809         C  E_AI_TANY  PC_ENT  AT     12.31          C
3811         C  E_AI_TANY  PC_ENT  BE     15.31          C
3812         C  E_AI_TANY  PC_ENT  BG      2.55          C
3813         C  E_AI_TANY  PC_ENT  CY      3.81          C
3814         C  E_AI_TANY  PC_ENT  CZ      6.01          C
...        ...        ...     ...  ..       ...        ...
221156    S951  E_AI_TANY  PC_ENT  PT      8.82          S
221157    S951  E_AI_TANY  PC_ENT  RO      0.00          S
221159    S951  E_AI_TANY  PC_ENT  SE      6.67          S
221160    S951  E_AI_TANY  PC_ENT  SI       NaN          S
221161    S951  E_AI_TANY  PC_ENT  SK      0.00          S

[1339 rows x 6 columns]


In [33]:
ai_adopt_2023.drop(columns=['nace_r2', 'indic_is', 'unit'], inplace=True) 
ai_adopt_2023.rename(columns={'nace_r2_1d' : 'nace_r2'})
print(ai_adopt_2023)

       geo  num_2023 nace_r2_1d
3809    AT     12.31          C
3811    BE     15.31          C
3812    BG      2.55          C
3813    CY      3.81          C
3814    CZ      6.01          C
...     ..       ...        ...
221156  PT      8.82          S
221157  RO      0.00          S
221159  SE      6.67          S
221160  SI       NaN          S
221161  SK      0.00          S

[1339 rows x 3 columns]


In [34]:
ai_adopt_2023 = ai_adopt_2023.dropna()
print(ai_adopt_2023)

       geo  num_2023 nace_r2_1d
3809    AT     12.31          C
3811    BE     15.31          C
3812    BG      2.55          C
3813    CY      3.81          C
3814    CZ      6.01          C
...     ..       ...        ...
221155  PL      9.33          S
221156  PT      8.82          S
221157  RO      0.00          S
221159  SE      6.67          S
221161  SK      0.00          S

[994 rows x 3 columns]


In [35]:
c = len(pd.unique(ai_adopt_2023['geo']))
print(f'Number of the available countries for the ai adoption variable {c}')

Number of the available countries for the ai adoption variable 28


In [27]:
### Wages

wages = pd.read_csv('C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/processed data/lc.csv', index_col=0)
print(wages)

     currency     unit sizeclas nace_r2 lcstruct geo      num_2016 flag_2016  \
freq                                                                           
A         EUR  P_SAL_H    10-49       B      D01  AL  2.500000e+00       NaN   
A         EUR  P_SAL_H    10-49       B      D01  AT  3.099000e+01       NaN   
A         EUR  P_SAL_H    10-49       B      D01  BA  3.910000e+00       NaN   
A         EUR  P_SAL_H    10-49       B      D01  BE  3.548000e+01       NaN   
A         EUR  P_SAL_H    10-49       B      D01  BG           NaN       NaN   
...       ...      ...      ...     ...      ...  ..           ...       ...   
A         PPS    TOTAL    TOTAL     S96    D1111  RS  3.134016e+07         d   
A         PPS    TOTAL    TOTAL     S96    D1111  SI  4.127085e+07         d   
A         PPS    TOTAL    TOTAL     S96    D1111  SK  3.383858e+07       NaN   
A         PPS    TOTAL    TOTAL     S96    D1111  TR  4.904631e+08       NaN   
A         PPS    TOTAL    TOTAL     S96 

In [37]:
un = pd.unique(wages['sizeclas'])
print(un)

['10-49' '250-499' '50-249' '500-999' 'GE10' 'GE1000' 'LT10' 'TOTAL']


In [ ]:
# Filter the data 
# D11 - wages and salaries 

wg_2016 = wages.copy()
wg_2020 = wages.copy()

ai_adopt_2023 = ai_adopt.drop(columns=[ 'freq', 'size_emp', 'num_2021', 'flag_2021', 'flag_2023', 'num_2024', 'flag_2024'])

ai_adopt_2023 = ai_adopt_2023[
    (ai_adopt_2023["indic_is"] == "E_AI_TANY") &
    (ai_adopt_2023["unit"] == "PC_ENT") 
]

ai_adopt_2023 = ai_adopt_2023[ai_adopt_2023['geo'].isin(EU_EFTA)
print(ai_adopt_2023)